[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter6/metrics.ipynb)

# Chapter 6: RAG Evaluation Metrics

This notebook demonstrates traditional NLP metrics for evaluating RAG systems, including context precision/recall, BLEU, and ROUGE scores.

**What you'll learn:**
- Calculate context precision and recall to evaluate retrieval quality
- Compute BLEU and ROUGE-L scores to evaluate generated answers
- Interpret metric results across examples with varying retrieval and generation quality

**Prerequisites:** `pip install spacy sacrebleu rouge` and run `python -m spacy download en_core_web_sm`.

In [1]:
import spacy
import sacrebleu
from rouge import Rouge

In [2]:
# Load the spacy model once to be used in the evaluation function.
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Spacy model 'en_core_web_sm' not found.")
    print("Please run: python -m spacy download en_core_web_sm")
    nlp = None

In [3]:
def evaluate_retrieval(retrieved_chunk_ids, relevant_chunk_ids):
    """
    Calculates Context Precision and Context Recall for the retrieval step.

    Args:
        retrieved_chunk_ids (list): A list of IDs of the chunks retrieved by the RAG system.
        relevant_chunk_ids (list): A list of ground truth IDs of all chunks relevant to the query.

    Returns:
        dict: A dictionary containing context_precision and context_recall.
    """
    retrieved_set = set(retrieved_chunk_ids)
    relevant_set = set(relevant_chunk_ids)

    if not retrieved_set and not relevant_set:
        return {"context_precision": 1.0, "context_recall": 1.0}
    
    if not retrieved_set:
        return {"context_precision": 0.0, "context_recall": 0.0}
        
    if not relevant_set:
        # Retrieved irrelevant chunks when none were relevant.
        return {"context_precision": 0.0, "context_recall": 1.0}

    # TP = true positives
    true_positives = len(retrieved_set.intersection(relevant_set))

    # Context Precision = TP / |retrieved|
    # Measures the signal-to-noise ratio of the retrieved context.
    context_precision = true_positives / len(retrieved_set)

    # Context Recall = TP / |relevant|
    # Measures how much of the necessary context was actually retrieved.
    context_recall = true_positives / len(relevant_set)

    return {"context_precision": context_precision, "context_recall": context_recall}

In [4]:
def evaluate_generation(generated_answer, ground_truth_answer):
    """
    Calculates BLEU and ROUGE-L scores for the generation step.

    Args:
        generated_answer (str): The answer generated by the RAG system.
        ground_truth_answer (str): The reference or "golden standard" answer.

    Returns:
        dict: A dictionary containing the BLEU and ROUGE-L f-measure scores.
    """
    # Calculate BLEU Score for standardized scoring.
    # sacrebleu expects a list of hypotheses and a list of lists of references.
    hypotheses = [generated_answer]
    references = [[ground_truth_answer]]
    
    # The score is on a 0-100 scale, so we divide by 100 to normalize to 0-1.
    bleu_score = sacrebleu.corpus_bleu(hypotheses, references).score / 100.0

    # Calculate ROUGE Score
    if not generated_answer.strip():
        rouge_scores = {'rouge-l': {'f': 0.0}}
    else:
        rouge = Rouge()
        rouge_scores = rouge.get_scores(generated_answer, ground_truth_answer)[0]

    return {
        "bleu": bleu_score,
        "rouge-l-f": rouge_scores['rouge-l']['f'] # F-measure
    }

In [5]:
evaluation_data = [
    # Basic Example
    {
        "query": "What is the function of the mitochondria and where are they found?",
        "retrieved_chunks": {
            "cell_bio_01": "Mitochondria are membrane-bound cell organelles (mitochondrion, singular) that generate most of the chemical energy needed to power the cell's biochemical reactions.",
            "cell_bio_02": "The cell nucleus contains the majority of the cell's genetic material in the form of multiple linear DNA molecules organized into structures called chromosomes.",
            "cell_bio_03": "These organelles are found in the cytoplasm of the cells of nearly all eukaryotic organisms, including animals, plants, and fungi."
        },
        "relevant_chunk_ids": ["cell_bio_01", "cell_bio_03"],
        "generated_answer": "Mitochondria generate chemical energy and are located in the cytoplasm of eukaryotic cells.",
        "ground_truth_answer": "The primary function of mitochondria is to generate chemical energy to power the cell, and they are found in the cytoplasm of eukaryotic cells."
    },
    # Example demonstrating failures in RAG - wrong chunks retrieved
    {
        "query": "Explain the process of photosynthesis.",
        "retrieved_chunks": {
            "plant_bio_05": "Cellular respiration is the process by which organisms combine oxygen with foodstuff molecules, diverting the chemical energy in these substances into life-sustaining activities.",
            "plant_bio_08": "A leaf is a principal appendage of the stem of a vascular plant, usually borne laterally aboveground and specialized for photosynthesis."
        },
        "relevant_chunk_ids": ["plant_bio_01", "plant_bio_02"], # Note: The system failed to retrieve the correct chunks
        "generated_answer": "A leaf is an appendage on a plant stem used for photosynthesis.", # Answer based on wrong context
        "ground_truth_answer": "Photosynthesis is the process used by plants, algae, and certain bacteria to harness energy from sunlight and turn it into chemical energy."
    },
    {
        "query": "Who was Marie Curie and what were her main achievements?",
        "retrieved_chunks": {
            "curie_bio_01": "Marie Skłodowska Curie (born Maria Salomea Skłodowska) was a Polish and naturalized-French physicist and chemist who conducted pioneering research on radioactivity.",
            "curie_bio_02": "Her achievements include the development of the theory of radioactivity, techniques for isolating radioactive isotopes, and the discovery of two elements, polonium and radium.",
            "curie_bio_03": "She was the first woman to win a Nobel Prize, the first person and the only woman to win the Nobel Prize twice, and the only person to win the Nobel Prize in two scientific fields."
        },
        "relevant_chunk_ids": ["curie_bio_01", "curie_bio_02", "curie_bio_03"],
        "generated_answer": "Marie Curie was a physicist who did research on radioactivity.", # Incomplete answer
        "ground_truth_answer": "Marie Curie was a physicist and chemist known for her pioneering research on radioactivity, discovering two elements (polonium and radium), and being the first woman to win a Nobel Prize."
    }
]

print("--- RAG Evaluation: Retrieval and Generation ---")
for i, item in enumerate(evaluation_data):
    print(f"\n--- Evaluating Item {i+1}: \"{item['query']}\" ---")
    
    retrieved_ids = list(item["retrieved_chunks"].keys())

    # 1. Evaluate Retrieval
    retrieval_metrics = evaluate_retrieval(retrieved_ids, item['relevant_chunk_ids'])
    print("[Retrieval Evaluation]")
    print(f"  Context Precision: {retrieval_metrics['context_precision']:.4f} (Measures retriever noise)")
    print(f"  Context Recall:    {retrieval_metrics['context_recall']:.4f} (Measures if all context was found)")

    # 2. Evaluate Generation
    generation_metrics = evaluate_generation(item['generated_answer'], item['ground_truth_answer'])
    print("[Generation Evaluation]")
    print(f"  BLEU Score:        {generation_metrics['bleu']:.4f} (Measures fluency/correctness)")
    print(f"  ROUGE-L Score:     {generation_metrics['rouge-l-f']:.4f} (Measures content similarity)")

--- RAG Evaluation: Retrieval and Generation ---

--- Evaluating Item 1: "What is the function of the mitochondria and where are they found?" ---
[Retrieval Evaluation]
  Context Precision: 0.6667 (Measures retriever noise)
  Context Recall:    1.0000 (Measures if all context was found)
[Generation Evaluation]
  BLEU Score:        0.2362 (Measures fluency/correctness)
  ROUGE-L Score:     0.6471 (Measures content similarity)

--- Evaluating Item 2: "Explain the process of photosynthesis." ---
[Retrieval Evaluation]
  Context Precision: 0.0000 (Measures retriever noise)
  Context Recall:    0.0000 (Measures if all context was found)
[Generation Evaluation]
  BLEU Score:        0.0162 (Measures fluency/correctness)
  ROUGE-L Score:     0.1250 (Measures content similarity)

--- Evaluating Item 3: "Who was Marie Curie and what were her main achievements?" ---
[Retrieval Evaluation]
  Context Precision: 1.0000 (Measures retriever noise)
  Context Recall:    1.0000 (Measures if all context w

> **Note:** These three examples illustrate different failure modes. <br>**Item 1** (mitochondria): perfect recall (1.0) means all relevant chunks were retrieved, but precision of 0.67 means one irrelevant chunk sneaked in; ROUGE-L of 0.65 shows good answer overlap. <br>**Item 2** (photosynthesis): zeros across retrieval metrics — the system retrieved entirely wrong chunks, so generation had no chance. <br>**Item 3** (Marie Curie): perfect retrieval (1.0/1.0) but low BLEU (0.05) because the generated answer was too brief despite having all the right context — a generation quality issue, not a retrieval one.